## Chunking

Chunking determines:

- What gets embedded
- What gets retrieved
- What context reaches the LLM

A bad chunking strategy can make a great embedding model fail.

There are 6 major chunking families:
- Fixed Size -> Split every N characters. Can break sentences
-  Overlapping
- Recursive -> Try large boundaries first. priority order: Paragraph, Sentence, Word, Character (It preserves semantic structure better than fixed-size chunking.)
- Token-Based Chunking -> LLMs think in tokens. Not characters.
- Structure Aware -> Instead of splitting based on:Characters, Tokens, Sentence length. you split based on the document's structure. 
- Semantic: Uses embeddings to determine boundaries. \
Advantages: Produces chunks that align with meaning. \
Disadvantages: Expensive. Requires embedding generation during ingestion. \
How it works: Split into sentences, Generate embeddings:, Compute similarity (S1 ↔ S2 = 0.92, S2 ↔ S3 = 0.89, , S3 ↔ S4 = 0.25, if threshold > 0.5 then creates a new chunk Chunk 1 = S1,S2,S3, Chunk 2 = S4) 
- Hierarchical


## When to Use What?

| Strategy        | Best For                         |
| --------------- | -------------------------------- |
| Fixed           | Small/simple docs                |
| Recursive       | Default RAG baseline             |
| Structure-aware | PDFs, docs, markdown, code       |
| Semantic        | Topic-heavy content              |
| Hierarchical    | Large enterprise knowledge bases |
| Parent-child    | Production RAG                   |
| RAPTOR          | Research-grade RAG               |


In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
file_path = "../data/pdf_files/Big-Book-of-Data-Engineering-Final.pdf"
loader = PyPDFLoader(
    file_path = file_path
)
documents = loader.load()
print(f"Number of documents loaded: {len(documents)}")

C:\Users\Aravindh\AppData\Local\Temp\ipykernel_27184\3395345364.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
f:\WeCloudData\AI\AgenticAI\langchain_ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents loaded: 57


In [1]:

def write_chunk(chunks, file_name):
    with open(f"../data/pdf_files/chunks/{file_name}", "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(chunk.page_content)
            f.write("\n")
            f.write("*" * 50)
            f.write("\n")

### Recursive chunking

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_splitter = RecursiveCharacterTextSplitter(
     chunk_size = 500,
     chunk_overlap = 50,
     length_function = len,
     separators = ["\n\n", ". ", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Successfully splitter {len(documents)} pdf documents into {len(chunks)} chunks")

write_chunk(chunks, "recursive_chunks.txt")

Successfully splitter 57 pdf documents into 315 chunks


### Token based chunk

In [6]:
from langchain_text_splitters import TokenTextSplitter 

text_splitter = TokenTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(f"Successfully splitter {len(documents)} pdf documents into {len(chunks)} chunks")

write_chunk(chunks, "token_based_chunks.txt")

Successfully splitter 57 pdf documents into 85 chunks


### Structure aware chunking

In [22]:
from unstructured.chunking import CHUNK_MAX_CHARS_DEFAULT
from unstructured.partition.pdf import partition_pdf 
from pdfminer.utils import open_filename

file_path = "../data/pdf_files/Big-Book-of-Data-Engineering-Final.pdf"
elements = partition_pdf(
    filename = file_path,
    strategy = "hi_res",
    infer_table_structure = True,
    # chunking_strategy= "by_title", # basic
    # max_characters = 500,
    # new_after_n_chars = 400,
    # combine_text_under_n_chars=200
)

# for i, element in enumerate(elements[:10]):
#     print(f"\n----- Element {i} -----")
#     print(type(element).__name__)
#     print(str(element))

with open("../data/pdf_files/chunks/basic_structure_base_chunk.txt", "w", encoding="utf-8") as f:
    for element in elements:
        f.write(element.text)

No languages specified, defaulting to English.


In [26]:
# chunk by title
file_path = "../data/pdf_files/Big-Book-of-Data-Engineering-Final.pdf"

elements = partition_pdf(
    filename = file_path,
    strategy = "hi_res",
    max_characters = 500,
    chunking_strategy = "by_title",
    combine_text_under_n_chars = 200,
    infer_table_structure = True 
)

with open("../data/pdf_files/chunks/title_based_structure_chunk_1.txt", "w", encoding="utf-8") as f:
    for element in elements:
        f.write(f"[{type(element).__name__}]\n")
        f.write(f"{element.text}\n")
        f.write("-" * 40 + "\n")  #

No languages specified, defaulting to English.


In [ ]:
# extarct image from pdf

from unstructured.partition.pdf import partition_pdf

file_path = "../data/pdf_files/bill_1.pdf"
elements = partition_pdf(
    filename = file_path,
    strategy = "hi_res",
    infer_table_structure = True,
    extract_images_in_pdf=True,
    languages = ["en"],
    extract_image_block_output_dir= "../data/pdf_files/image/",
    chunking_strategy=""
)

for element in elements:
    print(f"\n----- Element {i} -----")
    print(type(element).__name__)
    print(str(element))


----- Element 9 -----
Image
X GEM HOSPITAL

----- Element 9 -----
Title
GEM HOSPITAL

----- Element 9 -----
NarrativeText
45, PANKAJA MILLS ROAD, PALANIAPPA NAGAR, SOWRIPALAYAM PIRIVU,

----- Element 9 -----
Text
RAMANATHAPURAM, COIMBATORE - 641045

----- Element 9 -----
Image
Saccponson

----- Element 9 -----
NarrativeText
P: 0422 2325100; Email:info@geminstitute.in; WebSite:www.gemhospitals.com

----- Element 9 -----
NarrativeText
CIN: 33AABCG8302P2ZH

----- Element 9 -----
Text
FREE BILL

----- Element 9 -----
Title
R0 DT AR N

----- Element 9 -----
Text
Bill No / Date

----- Element 9 -----
Text
: 175587 / 31-12-2025 08:20

----- Element 9 -----
Text
Doctor/ Dept

----- Element 9 -----
Text
:

----- Element 9 -----
Text
Dr ANAND VIJAI N / SURGICAL GASTROENTEROLOGY

----- Element 9 -----
Text
Registration No

----- Element 9 -----
Text
: 10129810

----- Element 9 -----
Text
Patient Type

----- Element 9 -----
Text
: GENERAL

----- Element 9 -----
Text
Patient Name

----- Element 9 

### Semantic Chunking

from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker 
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename = file_path,
    strategy = "hi_res",
    infer_table_structure = True,
    max_characters = 500,
    combine_text_under_n_chars = 200,
)

text = "\n".join(element.text for element in elements if hasattr(element,"text") and element.text)


model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name = model_name)

splitter = SemanticChunker(embedding_model)

docs = splitter.create_documents([text])

print(f"Number of chunks: {len(docs)}")

for doc in docs[:5]:
    print(doc.page_content)
    print("=" * 80)

write_chunk(docs, "semantic_chunking.txt")

C:\Users\Aravindh\AppData\Local\Temp\ipykernel_27184\157908582.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
No languages specified, defaulting to English.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5050.84it/s]


Number of chunks: 33
A
Data Engineering The Big Book of ! e~ F collection of technical blogs, including code samples and notebooks I A A A > > >
< databricks
The Big Book of Data Engineering
Contents
secTion 1 Introduction to Data Engineering on Databricks . secTioN 2 Real-Life Use Cases on the Databricks Lakehouse Platform
...
2.1 Real-Time Point-of-Sale Analytics With the Data LakehousSe ... 9 2.2 Building a Cybersecurity Lakehouse for CrowdStrike Falcon Events ..., 14 2.3 Unlocking the Power of Health Data With a Modern Data Lakehouse................c.ccooooi 19 2.4 Timeliness and Reliability in the Transmission of Regulatory REPOrtS...........ccooooovvoooooiooeieeeeeeeeee 24 2.5 AML Solutions at Scale Using Databricks Lakehouse Platform ... 30 2.6 Build a Real-Time Al Model to Detect Toxic Behavior in Gaming ..., 41 2.7 Driving Transformation at Northwestern Mutual (Insights Platform) by Moving Toward a Scalable, Open Lakehouse ArchiteCture ... 44
2.8 How Databricks Data Team Built

In [4]:
len(docs)

33

In [ ]:
from sentence_transformers import SentenceTransformer

model_name: str = "all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)


ModuleNotFoundError: No module named 'langchain_experimental'